# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library and Python's data analytics stack.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and includes clinical variables from cancer survivors with second primary colorectal cancer.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Metadata is a custom object supporting .to_json()
metadata = dataset.metadata.to_json()

print(f"{getattr(dataset.metadata, 'name', 'Dataset')}: {getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** When working with Croissant datasets in `mlcroissant`, you should reference record sets and fields by their `@id`. We'll enumerate record sets and their fields here.

In [ ]:
# List all available record sets by @id
recordset_entities = dataset.metadata.recordSet

print("Available record sets:")
recordset_ids = []
if recordset_entities:
    for rs in recordset_entities:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        print(f"  - @id: {rs_id}\n    name: {rs_name}")
        recordset_ids.append(rs_id)
else:
    print("No record sets defined in metadata (schema may rely solely on distributions). Attempting to inspect available distributions as potential record sets.")
    # As a fallback, enumerate distributions (data files) with their @id
    if getattr(dataset.metadata, 'distribution', None):
        distributions = dataset.metadata.distribution
        for dist in distributions:
            dist_id = getattr(dist, '@id', None)
            print(f"  - Distribution @id: {dist_id}")
        recordset_ids = [getattr(dist, '@id', None) for dist in distributions]
    else:
        print("No distributions found either.")

# For each record set, list its fields (if possible)
if recordset_entities:
    for rs in recordset_entities:
        print(f"\nFields in record set {getattr(rs, '@id', None)}:")
        if getattr(rs, 'field', None):
            for fld in rs.field:
                fld_id = getattr(fld, '@id', None)
                fld_name = getattr(fld, 'name', None)
                print(f"  - @id: {fld_id}\n    name: {fld_name}")
        else:
            print("  (No fields defined)")

## 3. Data Extraction
Load data from available record set(s) into pandas DataFrames for inspection.

We use record set or distribution `@id`s from the overview above.

In [ ]:
# If no recordSet entities in the Croissant metadata, we'll attempt to load from the available distribution @id

# Use IDs found in the previous step
# For this dataset, recordset_ids likely empty, so we'll use distribution IDs instead
if not recordset_ids:
    distributions = getattr(dataset.metadata, 'distribution', None)
    if distributions:
        recordset_ids = [getattr(dist, '@id', None) for dist in distributions]
    else:
        print("No accessible record sets or distributions!")
        recordset_ids = []

# For this dataset, the main tabular data is in this distribution ID:
main_tabular_id = 'http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7862866/_/62935be8-24c5-4111-9be6-0b2e3d9593bd'
if main_tabular_id in recordset_ids:
    selected_ids = [main_tabular_id]
else:
    selected_ids = recordset_ids

dataframes = {}
for rs_id in selected_ids:
    try:
        print(f"\nReading data for record set/distribution @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print("Columns:", dataframes[rs_id].columns.tolist())
            display(dataframes[rs_id].head())
        else:
            print("  (No records loaded or record set empty)")
    except Exception as e:
        print(f"  (Error extracting records: {e})")

## 4. Exploratory Data Analysis (EDA)
We'll process the extracted DataFrame by filtering, normalizing numeric fields, and grouping the data.

**Note:** We reference all columns/fields by their `@id` where possible. Inspect the columns to choose meaningful fields for demonstration.

In [ ]:
# Pick the primary DataFrame (if more than one, the main tabular @id)
df_key = main_tabular_id if main_tabular_id in dataframes else next(iter(dataframes.keys()))
df = dataframes[df_key]

print('Available columns:', list(df.columns))

# Choose a numeric field for demo, e.g. 'Age' or similar (check for a likely column)
numeric_candidates = [col for col in df.columns if 'age' in col.lower() or df[col].dtype.kind in 'if']

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = df.columns[0]  # fallback
print(f"Chosen numeric field: {numeric_field_id}")

# Some columns may be string-typed but numeric in value. Attempt conversion:
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean()  # demo: use mean as threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered rows where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id}:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a (likely) categorical field. Try 'Sex', 'Gender', or 'MSI' if present.
group_candidates = [col for col in df.columns if any(
    phrase in col.lower() for phrase in ['sex', 'gender', 'msi', 'status', 'group']
)]
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by: {group_field}")

    # For aggregation, ignore non-numeric columns except the normalized/selected:
    agg_fields = filtered_df.select_dtypes(include=[np.number]).columns.intersection([numeric_field_id, f"{numeric_field_id}_normalized"])
    grouped_df = filtered_df.groupby(group_field)[agg_fields].mean()
    print(grouped_df.head())
else:
    print("No suitable group field found in columns.")

## 5. Visualization
Visualize distributions or relationships between fields. For example, plot the normalized numeric field or compare groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Visualize group comparison if a group field was found
if 'group_field' in locals() and group_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

- This notebook demonstrated loading clinical data from the FAIR² colorectal cancer survivors dataset using Croissant schema with `mlcroissant`.
- We explored available data structures by their `@id`, loaded the table(s), and performed simple EDA using pandas and visualization tools.
- By referencing fields and record sets by their `@id`, this workflow ensures robust and transparent dataset handling.
- The dataset enables studies on predictors and distributions of MSI-H phenotype in a clinical survivor cohort.